5.1. Implement subword-level tokenization using a BPE-based tokenizer.
Tokenize the sentence into subword units.
Display the generated tokens and their corresponding token IDs.
(Implement two separate codes: a. Using pretrained model and b. Without using pretrained model)


In [1]:
!pip install transformers


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 5.1(a)
from transformers import GPT2Tokenizer

# Load pretrained BPE tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Input sentence
text = "Natural language processing is a field of artificial intelligence."
print("Input:", text)

# Tokenize into subword units
tokens = tokenizer.tokenize(text)

# Convert tokens to token IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("\nToken\t\tToken ID")
print("-" * 35)

for token, token_id in zip(tokens, token_ids):
    print(f"{token:20} {token_id}")

D:\sem7\NLP Lab\nlp_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


D:\sem7\NLP Lab\nlp_venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP OMNIBOOK\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Input: Natural language processing is a field of artificial intelligence.

Token		Token ID
-----------------------------------
Natural              35364
Ġlanguage            3303
Ġprocessing          7587
Ġis                  318
Ġa                   257
Ġfield               2214
Ġof                  286
Ġartificial          11666
Ġintelligence        4430
.                    13


In [3]:
#5.1(b)
import re
from collections import Counter

# Read corpus
with open("input.txt", "r", encoding="utf-8") as file:
    text = file.read().lower()

# Extract words
words = re.findall(r'\b[a-z]+\b', text)

# Count word frequencies
word_freq = Counter(words)

# Represent each word as characters
vocab = {}

for word, freq in word_freq.items():
    vocab[tuple(list(word) + ["</w>"])] = freq


# Find frequencies of adjacent pairs
def get_pairs(vocab):
    pairs = Counter()

    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq

    return pairs


# Merge the most frequent pair
def merge_pair(pair, vocab):
    new_vocab = {}

    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0

        while i < len(symbols):
            if (i < len(symbols) - 1 and
                    symbols[i] == pair[0] and
                    symbols[i + 1] == pair[1]):

                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        new_vocab[tuple(new_symbols)] = freq

    return new_vocab


# Learn BPE merges
num_merges = 100
merges = []

for i in range(num_merges):

    pairs = get_pairs(vocab)

    if not pairs:
        break

    best_pair = pairs.most_common(1)[0][0]

    merges.append(best_pair)

    vocab = merge_pair(best_pair, vocab)


print("Learned BPE Merges:")
for i, pair in enumerate(merges, 1):
    print(i, pair)


# Tokenize a new word
def tokenize_word(word, merges):

    symbols = list(word.lower()) + ["</w>"]

    for pair in merges:

        new_symbols = []
        i = 0

        while i < len(symbols):

            if (i < len(symbols) - 1 and
                    symbols[i] == pair[0] and
                    symbols[i + 1] == pair[1]):

                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2

            else:
                new_symbols.append(symbols[i])
                i += 1

        symbols = new_symbols

    return symbols


# Create subword vocabulary
subword_vocab = set()

for symbols in vocab:
    for symbol in symbols:
        subword_vocab.add(symbol)

# Assign token IDs
token_to_id = {
    token: i
    for i, token in enumerate(sorted(subword_vocab))
}


# Test sentence
sentence = "Natural language processing is a field of artificial intelligence."
print("Input:", sentence)

print("\nToken\t\tToken ID")
print("-" * 35)

for word in re.findall(r'\b[a-z]+\b', sentence.lower()):

    tokens = tokenize_word(word, merges)

    for token in tokens:

        token_id = token_to_id.get(token, -1)

        print(f"{token:20} {token_id}")

Learned BPE Merges:
1 ('e', '</w>')
2 ('s', '</w>')
3 ('e', 'n')
4 ('i', 'n')
5 ('d', '</w>')
6 ('e', 'r')
7 ('o', 'r')
8 ('a', 'n')
9 ('t', 'h')
10 ('o', 'n')
11 ('a', 't')
12 ('a', 'r')
13 ('r', 'e')
14 ('a', 'l')
15 ('in', 'g')
16 ('t', 'o')
17 ('y', '</w>')
18 ('ing', '</w>')
19 ('en', 't')
20 ('i', 'on')
21 ('th', 'e</w>')
22 ('w', 'or')
23 ('l', 'e')
24 ('at', 'ion')
25 ('er', '</w>')
26 ('e', 'd</w>')
27 ('to', 'k')
28 ('tok', 'en')
29 ('a', '</w>')
30 ('u', 'n')
31 ('e', 's</w>')
32 ('an', 'd</w>')
33 ('o', 'c')
34 ('t', '</w>')
35 ('i', 'z')
36 ('s', 'u')
37 ('wor', 'd</w>')
38 ('al', '</w>')
39 ('ation', '</w>')
40 ('q', 'u')
41 ('an', '</w>')
42 ('c', 'h')
43 ('i', 'c')
44 ('p', 'r')
45 ('e', 's')
46 ('f', '</w>')
47 ('u', 'l')
48 ('c', 'an</w>')
49 ('o', 'f</w>')
50 ('to', '</w>')
51 ('ent', '</w>')
52 ('o', 'm')
53 ('s', 't')
54 ('u', 'r')
55 ('g', 'e</w>')
56 ('i', 's</w>')
57 ('e', 'l')
58 ('p', 're')
59 ('le', 'ar')
60 ('lear', 'n')
61 ('token', 'iz')
62 ('s', 'e')
63 (